In [1]:
import os
import time
import glob
import numpy as np
import pandas as pd
import datetime as datetime

from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [2]:
conn = presto.connect(
        host="presto-gateway.serving.data.plectrum.dev",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )

presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

In [3]:
local_dataset_path = '/Users/E2074/local-datasets/customer/loyalty/experiment2/'

In [5]:
start_date = '20250512'
end_date = '20250518'
city = {'Delhi' : 'loyalty_experiment2_delhi', 
        'Bangalore' : 'loyalty_experiment2_bangalore', 
        'Hyderabad' : 'loyalty_experiment2_hyd'}

experiment_name = 'loyalty_experiment2'

In [8]:
start = datetime.strptime(start_date, '%Y%m%d')
end = datetime.strptime(end_date, '%Y%m%d')

date_list = [(start + timedelta(days=i)).strftime('%Y%m%d') for i in range((end - start).days + 1)]

print(date_list)

['20250512', '20250513', '20250514', '20250515', '20250516', '20250517', '20250518']


In [12]:
for date in date_list[6:]:
    
    print( '---------' , date , '---------')
    
    for city_name, execution_name in  city.items():
        
        print('Doing Science for', city_name)
        
        query = f"""
            
            with rapidex as (
            
            select 
                distinct
                sample_category,
                customer_id
            from 
                iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
            where 
                yyyymmdd >= '20250520'
                and yyyymmdd <= '20250526'
                and experiment_name = '{experiment_name}'
                and execution_name = '{execution_name}'
            ),
            
            orders as (
        
            select
                yyyymmdd,
                customer_id,
                city_name,
                service_obj_service_name service_name,
                order_id,
                modified_order_status,
                customer_cancelled_epoch,
                order_requested_epoch,
                accepted_epoch,
                accept_to_cancelled,
                case when modified_order_status = 'COBRA' then ((customer_cancelled_epoch - order_requested_epoch)/1000) end cobra_ttc,
                case when modified_order_status = 'OCARA' then ((customer_cancelled_epoch - accepted_epoch)/1000) end ocara_ttc,
                case when accepted_epoch is not null then ((accepted_epoch - order_requested_epoch)/1000) end tta,
                epoch,
                quarter_hour,
                hhmmss,
                distance_final_distance,
                accept_to_pickup_distance,
                pickup_location_hex_12,
                drop_location_hex_12,
                pickup_location_hex_8,
                drop_location_hex_8,
                channel_host,
                case 
                    when cast(substr(quarter_hour,1,2) as real) > 7 and cast(substr(quarter_hour,1,2) as real) <= 11
                        then 'morning_peak'
                    when cast(substr(quarter_hour,1,2) as real) > 11 and cast(substr(quarter_hour,1,2) as real) <= 16
                        then 'afternoon'
                    when cast(substr(quarter_hour,1,2) as real) > 16 and cast(substr(quarter_hour,1,2) as real) <= 21
                        then 'evening_peak'
                    when cast(substr(quarter_hour,1,2) as real) > 21 and cast(substr(quarter_hour,1,2) as real) <= 23
                        then 'rest_evening'
                    when cast(substr(quarter_hour,1,2) as real) >= 0 and cast(substr(quarter_hour,1,2) as real) <= 7
                        then 'rest_morning'
                end as time_bucket
            from 
                orders.order_logs_fact a
            where 
                yyyymmdd = '{date}'
                and city_name = '{city_name}'
                and service_obj_service_name in ('Auto', 'Link', 'CabAC', 'Auto Pool', 'CabEconomy', 'CabPremium', 'Cab SUV', 'Bike Lite', 'Auto Pet', 'Auto NCR', 'Bike Metro', 'Scooty')
                and lower(channel_host) in ('android', 'ios')
            )
        
            select 
                a.sample_category,
                b.*
            from 
                rapidex a
            join 
                orders b
                on a.customer_id = b.customer_id
        """
    
        df_query = pd.read_sql(query, conn3)
        df_query.to_parquet(local_dataset_path + city_name + '/preperiod/{}_{}_orders_data_dump.parquet'.format(date, city_name), index=False)

--------- 20250518 ---------
Doing Science for Delhi
Doing Science for Bangalore
Doing Science for Hyderabad
